# RQ2 and RQ4 — same 25-pair gold as Table 4

Colab: Runtime → Run all.

- **RQ2:** connected components of scaffold edges at threshold τ vs connected components of the 25 gold edges. Adjusted Rand Index + purity. Gold is not used to build the predicted clusters.
- **RQ4:** each of the 937 cross-domain pairs gets a status from the *score and theme structure only*. Gold is applied only when computing precision-by-status.

Same 50 labels, same theme lexicon, same `0.65 * theme Jaccard + 0.35 * token Jaccard` as `scaffold_vs_minilm`.

In [ ]:
from __future__ import annotations
from collections import defaultdict, Counter
from itertools import combinations
from dataclasses import dataclass
import csv, os

def adjusted_rand_score(y_true, y_pred):
    """ARI without sklearn. Same definition as sklearn.metrics.adjusted_rand_score."""
    from math import comb
    n = len(y_true)
    if n < 2:
        return 1.0
    ct = {}
    for a, b in zip(y_true, y_pred):
        ct[(a, b)] = ct.get((a, b), 0) + 1
    sum_comb_c = sum(comb(v, 2) for v in Counter(y_true).values())
    sum_comb_k = sum(comb(v, 2) for v in Counter(y_pred).values())
    sum_comb = sum(comb(v, 2) for v in ct.values())
    total = comb(n, 2)
    expected = (sum_comb_c * sum_comb_k) / total if total else 0.0
    max_index = 0.5 * (sum_comb_c + sum_comb_k)
    if max_index == expected:
        return 1.0 if sum_comb == sum_comb_c == sum_comb_k else 0.0
    return (sum_comb - expected) / (max_index - expected)

OUT = "evaluation_output"
os.makedirs(OUT, exist_ok=True)

## Locked data (do not edit)

In [ ]:
CONCEPTS = [
    ("pesticide residue contamination", "Agriculture"),
    ("chemical contaminant bioaccumulation", "Environment"),
    ("agricultural water use", "Agriculture"),
    ("groundwater aquifer depletion", "Environment"),
    ("pollinator habitat loss", "Agriculture"),
    ("biodiversity species extinction", "Environment"),
    ("soil carbon sequestration", "Agriculture"),
    ("atmospheric carbon dioxide concentration", "Environment"),
    ("agroforestry land use", "Agriculture"),
    ("deforestation tropical forest", "Environment"),
    ("aquaculture fish farming", "Agriculture"),
    ("antibiotic resistance surveillance", "Healthcare"),
    ("food supply chain logistics", "Agriculture"),
    ("pediatric nutrition intervention", "Healthcare"),
    ("plant disease surveillance", "Agriculture"),
    ("public health surveillance system", "Healthcare"),
    ("seed genetic modification", "Agriculture"),
    ("precision medicine genomic targeting", "Healthcare"),
    ("farm subsidy policy", "Agriculture"),
    ("income inequality Gini coefficient", "Socioeconomics"),
    ("crop yield variability", "Agriculture"),
    ("food insecurity household level", "Socioeconomics"),
    ("agricultural labor migration", "Agriculture"),
    ("rural urban economic divide", "Socioeconomics"),
    ("rural smallholder income", "Agriculture"),
    ("poverty alleviation microfinance", "Socioeconomics"),
    ("air quality particulate matter", "Environment"),
    ("chronic disease self-management", "Healthcare"),
    ("urban heat island effect", "Environment"),
    ("emergency department crowding", "Healthcare"),
    ("drought frequency projection", "Environment"),
    ("maternal mortality risk factor", "Healthcare"),
    ("freshwater availability stress", "Environment"),
    ("infectious disease contact tracing", "Healthcare"),
    ("carbon emission trading", "Environment"),
    ("carbon tax distributional effect", "Socioeconomics"),
    ("renewable energy transition", "Environment"),
    ("labor market automation displacement", "Socioeconomics"),
    ("ecosystem restoration practice", "Environment"),
    ("climate adaptation funding equity", "Socioeconomics"),
    ("sea level rise projection", "Environment"),
    ("disaster recovery economic resilience", "Socioeconomics"),
    ("social determinants of health", "Healthcare"),
    ("neighborhood segregation index", "Socioeconomics"),
    ("health insurance coverage gap", "Healthcare"),
    ("health expenditure household burden", "Socioeconomics"),
    ("mental health service access", "Healthcare"),
    ("unemployment benefit duration", "Socioeconomics"),
    ("health equity racial disparity", "Healthcare"),
    ("racial discrimination employment", "Socioeconomics"),
]
DOMAIN = {n: d for n, d in CONCEPTS}
NAMES = [n for n, _ in CONCEPTS]
assert len(NAMES) == 50

GOLD_LIST = [
    ("pesticide residue contamination", "chemical contaminant bioaccumulation"),
    ("agricultural water use", "groundwater aquifer depletion"),
    ("pollinator habitat loss", "biodiversity species extinction"),
    ("soil carbon sequestration", "atmospheric carbon dioxide concentration"),
    ("agroforestry land use", "deforestation tropical forest"),
    ("aquaculture fish farming", "antibiotic resistance surveillance"),
    ("food supply chain logistics", "pediatric nutrition intervention"),
    ("plant disease surveillance", "public health surveillance system"),
    ("seed genetic modification", "precision medicine genomic targeting"),
    ("farm subsidy policy", "income inequality Gini coefficient"),
    ("crop yield variability", "food insecurity household level"),
    ("agricultural labor migration", "rural urban economic divide"),
    ("rural smallholder income", "poverty alleviation microfinance"),
    ("air quality particulate matter", "chronic disease self-management"),
    ("urban heat island effect", "emergency department crowding"),
    ("drought frequency projection", "maternal mortality risk factor"),
    ("freshwater availability stress", "infectious disease contact tracing"),
    ("carbon emission trading", "carbon tax distributional effect"),
    ("renewable energy transition", "labor market automation displacement"),
    ("ecosystem restoration practice", "climate adaptation funding equity"),
    ("sea level rise projection", "disaster recovery economic resilience"),
    ("social determinants of health", "neighborhood segregation index"),
    ("health insurance coverage gap", "health expenditure household burden"),
    ("mental health service access", "unemployment benefit duration"),
    ("health equity racial disparity", "racial discrimination employment"),
]
GOLD = {tuple(sorted(p)) for p in GOLD_LIST}
assert len(GOLD) == 25
assert all(DOMAIN[a] != DOMAIN[b] for a, b in GOLD_LIST)

THEME_LEX = {
    "contaminant": {"pesticide", "residue", "contamination", "chemical", "contaminant", "bioaccumulation", "particulate", "air"},
    "water": {"water", "groundwater", "aquifer", "freshwater", "irrigation", "drought", "aquaculture"},
    "biodiversity": {"pollinator", "habitat", "biodiversity", "species", "extinction", "ecosystem"},
    "carbon": {"carbon", "sequestration", "atmospheric", "dioxide", "emission", "trading"},
    "land_forest": {"agroforestry", "land", "deforestation", "tropical", "forest", "soil", "restoration"},
    "amr_onehealth": {"aquaculture", "antibiotic", "resistance", "surveillance", "disease", "infectious", "public", "health", "plant"},
    "food_nutrition": {"food", "supply", "nutrition", "pediatric", "yield", "insecurity", "crop"},
    "genomics": {"seed", "genetic", "modification", "precision", "medicine", "genomic"},
    "income": {"farm", "subsidy", "income", "inequality", "gini", "poverty", "microfinance", "smallholder", "expenditure"},
    "labor": {"labor", "migration", "unemployment", "automation", "employment", "displacement", "market"},
    "rural_urban": {"rural", "urban", "divide", "neighborhood", "segregation"},
    "heat_emergency": {"heat", "island", "emergency", "department", "crowding"},
    "climate_disaster": {"drought", "sea", "level", "disaster", "recovery", "resilience", "climate", "adaptation"},
    "health_access": {"insurance", "coverage", "mental", "health", "equity", "racial", "disparity", "determinants", "mortality", "chronic", "disease"},
    "race": {"racial", "discrimination", "equity", "segregation", "neighborhood"},
    "energy": {"renewable", "energy", "transition", "automation", "labor"},
}
STOP = {"of", "the", "and", "a", "an", "to", "for", "in", "on", "level", "system", "practice", "effect", "use", "gap"}

def tokens(name):
    return {t for t in name.lower().replace("-", " ").split() if t not in STOP and len(t) > 2}

def themes(name):
    tok = tokens(name)
    return {th for th, lex in THEME_LEX.items() if tok & lex}

def scaffold_score(a, b):
    ta, tb = themes(a), themes(b)
    ka, kb = tokens(a), tokens(b)
    j_th = len(ta & tb) / len(ta | tb) if (ta | tb) else 0.0
    j_tok = len(ka & kb) / len(ka | kb) if (ka | kb) else 0.0
    conf = 0.65 * j_th + 0.35 * j_tok
    if not (ta & tb) and not (ka & kb):
        conf *= 0.15
    return conf, ta, tb, ka, kb, j_th, j_tok

print("concepts", len(NAMES))
print("domains", Counter(DOMAIN.values()))
print("gold", len(GOLD))

## Score every cross-domain pair (gold not used)

In [ ]:
rows = []
for a, b in combinations(NAMES, 2):
    if DOMAIN[a] == DOMAIN[b]:
        continue
    conf, ta, tb, ka, kb, j_th, j_tok = scaffold_score(a, b)
    rows.append({
        "a": a, "b": b,
        "da": DOMAIN[a], "db": DOMAIN[b],
        "score": conf,
        "shared_themes": tuple(sorted(ta & tb)),
        "n_shared_themes": len(ta & tb),
        "j_th": j_th,
        "j_tok": j_tok,
        "gold": tuple(sorted((a, b))) in GOLD,
    })
assert len(rows) == 937
print("cross-domain pairs", len(rows))
print("score min/median/max", min(r["score"] for r in rows), sorted(r["score"] for r in rows)[len(rows)//2], max(r["score"] for r in rows))

## RQ2 — group formation

Gold groups = connected components of the 25 gold edges (isolates are singleton groups).
Predicted groups at τ = connected components of scaffold edges with score ≥ τ.
Compare partitions with adjusted Rand index and cluster purity (majority-gold-label per predicted component).

In [ ]:
class UF:
    def __init__(self, nodes):
        self.p = {x: x for x in nodes}
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[rb] = ra
    def labels(self, nodes):
        roots = {}
        out = []
        k = 0
        for n in nodes:
            r = self.find(n)
            if r not in roots:
                roots[r] = k
                k += 1
            out.append(roots[r])
        return out, k

def partition_from_edges(nodes, edges):
    uf = UF(nodes)
    for a, b in edges:
        uf.union(a, b)
    return uf.labels(nodes)

gold_edges = list(GOLD)
gold_lab, n_gold_comp = partition_from_edges(NAMES, gold_edges)
print(f"Gold connected components (incl. singletons): {n_gold_comp}")
print(f"Gold non-singleton groups: {sum(1 for c, n in Counter(gold_lab).items() if n > 1)}")

def purity(pred_lab, gold_lab):
    by = defaultdict(list)
    for p, g in zip(pred_lab, gold_lab):
        by[p].append(g)
    correct = 0
    for members in by.values():
        correct += Counter(members).most_common(1)[0][1]
    return correct / len(pred_lab)

print()
print(f"{'τ':>6} {'edges':>7} {'pred_cc':>8} {'ARI':>8} {'purity':>8} {'non-sing':>9}")
rq2_rows = []
for tau in (0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60, 0.70):
    e = [(r["a"], r["b"]) for r in rows if r["score"] >= tau]
    pred_lab, ncc = partition_from_edges(NAMES, e)
    ari = adjusted_rand_score(gold_lab, pred_lab)
    pur = purity(pred_lab, gold_lab)
    ns = sum(1 for c, n in Counter(pred_lab).items() if n > 1)
    print(f"{tau:6.2f} {len(e):7d} {ncc:8d} {ari:8.3f} {pur:8.3f} {ns:9d}")
    rq2_rows.append((tau, len(e), ncc, ari, pur, ns))

with open(f"{OUT}/rq2_clustering.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["tau", "pred_edges", "pred_components", "ARI", "purity", "non_singleton_pred_groups"])
    w.writerows(rq2_rows)
print("\nWrote", f"{OUT}/rq2_clustering.csv")

## RQ4 — status without looking at gold

Status is a function of score and theme promiscuity only.

| Status | Rule (gold-free) |
|---|---|
| REJECTED | score < 0.10 and no shared theme |
| PROPOSED | 0.10 ≤ score < 0.25 |
| MATCHED | 0.25 ≤ score < 0.40 |
| CANDIDATE | 0.40 ≤ score < 0.50 |
| CONFOUNDED | score ≥ 0.40 and the pair’s shared themes are *promiscuous* (a shared theme also hits ≥ 8 other cross-domain partners at score ≥ 0.40) |
| VALIDATED | 0.50 ≤ score < 0.70 and not CONFOUNDED |
| CONFIRMED | score ≥ 0.70 and not CONFOUNDED |
| DEPRECATED | unused on a single snapshot |

Precision by status = gold fraction among pairs that received that status. Gold is not an input to the rule.

In [ ]:
# Promiscuous themes: gold-free. Count how many score≥0.40 pairs carry each theme.
theme_hit = Counter()
for r in rows:
    if r["score"] >= 0.40:
        for th in r["shared_themes"]:
            theme_hit[th] += 1
PROMISCUOUS = {th for th, n in theme_hit.items() if n >= 8}
print("Theme frequency at score≥0.40:", dict(theme_hit))
print("Promiscuous themes (≥8 pairs):", sorted(PROMISCUOUS))

def assign_status(r):
    s = r["score"]
    shared = set(r["shared_themes"])
    confounded = s >= 0.40 and bool(shared & PROMISCUOUS)
    if confounded:
        return "CONFOUNDED"
    if s < 0.10 and r["n_shared_themes"] == 0:
        return "REJECTED"
    if s < 0.10:
        return "PROPOSED"
    if s < 0.25:
        return "PROPOSED"
    if s < 0.40:
        return "MATCHED"
    if s < 0.50:
        return "CANDIDATE"
    if s < 0.70:
        return "VALIDATED"
    return "CONFIRMED"

for r in rows:
    r["status"] = assign_status(r)

print()
print(f"{'status':<14} {'n':>6} {'gold':>6} {'precision':>10} {'recall':>8}")
print("-" * 48)
order = ["CONFIRMED", "VALIDATED", "CANDIDATE", "CONFOUNDED", "MATCHED", "PROPOSED", "REJECTED", "DEPRECATED"]
rq4_rows = []
for st in order:
    subset = [r for r in rows if r["status"] == st]
    n = len(subset)
    g = sum(1 for r in subset if r["gold"])
    prec = g / n if n else float("nan")
    rec = g / 25.0
    prec_s = f"{prec:.3f}" if n else "—"
    print(f"{st:<14} {n:6d} {g:6d} {prec_s:>10} {rec:8.3f}")
    rq4_rows.append((st, n, g, prec if n else None, rec))

print("\nTotal gold recovered across statuses:", sum(1 for r in rows if r["gold"]), "/ 25")
assert sum(1 for r in rows if r["gold"]) == 25

with open(f"{OUT}/rq4_status_precision.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["status", "n_pairs", "gold_in_status", "precision", "recall_of_25"])
    for st, n, g, prec, rec in rq4_rows:
        w.writerow([st, n, g, "" if prec is None else f"{prec:.6f}", f"{rec:.6f}"])

with open(f"{OUT}/rq4_all_pairs.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["a", "b", "da", "db", "score", "status", "gold", "n_shared_themes"])
    w.writeheader()
    for r in sorted(rows, key=lambda x: -x["score"]):
        w.writerow({
            "a": r["a"], "b": r["b"], "da": r["da"], "db": r["db"],
            "score": f"{r['score']:.4f}", "status": r["status"],
            "gold": int(r["gold"]), "n_shared_themes": r["n_shared_themes"],
        })
print("Wrote", f"{OUT}/rq4_status_precision.csv", "and", f"{OUT}/rq4_all_pairs.csv")

## How to put this in the paper

- RQ2: report ARI and purity at the same τ = 0.50 used in Table 4. Say gold groups are the connected components of the 25 annotated edges.
- RQ4: report the precision-by-status table. State the status rule in Method so it is reproducible. Do not call CONFIRMED “clinically true.”